# PyTorch: Tensors & Autograd

**Interview focus:** tensor operations, `requires_grad`, computation graph, backward pass, `detach()`, and GPU transfer.

In [ ]:
import torch
import numpy as np

## 1. Tensor Creation & Basics

In [ ]:
# From data
t = torch.tensor([1, 2, 3], dtype=torch.float32)
t2 = torch.tensor([[1, 2], [3, 4]])

# Constructors
zeros = torch.zeros(3, 4)
ones = torch.ones(2, 3)
rand = torch.randn(3, 3)
arange = torch.arange(0, 10, 2)
eye = torch.eye(3)

# From NumPy (shares memory if float64/int64)
np_arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(np_arr)  # shares memory
np_back = t_from_np.numpy()           # shares memory

print(t.shape, t.dtype, t.device)

## 2. Tensor Operations

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(4, 5)

print((a @ b).shape)          # matrix multiply
print(a.T.shape)              # transpose
print(a.reshape(2, 6).shape)
print(a.unsqueeze(0).shape)     # add dim: (1, 3, 4)
print(a.squeeze().shape)      # remove size-1 dims

# Indexing works like NumPy
print(a[0], a[:, 1], a[0:2, 1:3])

## 3. Autograd — The Computation Graph

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1  # y = x^2 + 2x + 1

y.backward()  # dy/dx = 2x + 2 = 8
print(f"dy/dx at x=3: {x.grad}")

In [ ]:
# Vector-valued functions: pass gradient_weights to backward
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2  # element-wise

y.backward(torch.tensor([1.0, 1.0, 1.0]))  # sum of gradients
print(f"grad: {x.grad}")  # [2, 4, 6]

In [ ]:
# Chain rule in action
x = torch.tensor(2.0, requires_grad=True)
a = x * 3       # a = 6
b = a + 1       # b = 7
c = b ** 2      # c = 49

c.backward()
print(f"dc/dx = {x.grad}")  # 2*b*3 = 42

## 4. `detach()`, `no_grad()`, and In-place Ops

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
y = x ** 2
z = y.detach()  # z is NOT connected to the graph
w = z * 2

w.backward()
print(f"x.grad: {x.grad}")  # None — w doesn't depend on x through graph

x.grad = None
with torch.no_grad():
  y2 = x ** 2 * 3  # no graph built
print(f"y2.requires_grad: {y2.requires_grad}")

In [ ]:
# In-place ops break the graph for that version of the tensor
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2
x.add_(1)  # in-place — y's computation used old x
# y.backward() would give wrong gradients — avoid in-place on leaf tensors

## 5. GPU Transfer

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

x = torch.randn(3, 3).to(device)
y = torch.randn(3, 3, device=device)
z = x @ y  # stays on same device
print(z.device)

## 6. Practice Problems

In [ ]:
# Problem: Compute gradient of f(x) = sum(x_i^3) manually and with autograd
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
f = (x ** 3).sum()
f.backward()
print(f"Autograd: {x.grad}")       # [3, 12, 27]
print(f"Manual:   {3 * x.detach() ** 2}")

In [ ]:
# Problem: Implement softmax with autograd
def softmax(x):
  x_shifted = x - x.max(dim=-1, keepdim=True).values
  exp_x = torch.exp(x_shifted)
  return exp_x / exp_x.sum(dim=-1, keepdim=True)

logits = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=True)
probs = softmax(logits)
loss = -torch.log(probs[0, 2])  # negative log prob of class 2
loss.backward()
print(f"probs: {probs.detach()}")
print(f"grad:  {logits.grad}")